In [5]:
import surprise

import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\filmtrust\ratings.txt",
                 sep=' ',names=['UserID','ItemID','Rating'])
df.sample(5)

,UserID,ItemID,Rating
21560,896,235,4.0
11710,482,6,2.0
7008,293,7,4.0
32268,1375,254,2.0
4961,205,250,4.0


In [9]:
lowest_rating = df['Rating'].min()
higest_rating = df['Rating'].max()
lowest_rating,higest_rating

(np.float64(0.5), np.float64(4.0))

In [13]:
reader = surprise.Reader(rating_scale=(lowest_rating,higest_rating))
data = surprise.Dataset.load_from_df(df,reader)
type(data)

surprise.dataset.DatasetAutoFolds

## 'user_based': False -> item_based collaborative System

In [32]:
similarity_options = {'name':'cosine','user_based':True} 
algo = surprise.KNNBasic(sim_options = similarity_options, k=70) # default k=40
output = algo.fit(data.build_full_trainset())
# Calculates the  expected ratings for all the users 

Computing the cosine similarity matrix...
Done computing similarity matrix.


### Train Test Split

In [33]:
from surprise.model_selection import train_test_split

In [34]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=26)
algo.fit(trainset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


In [35]:
pred = algo.test(testset)
surprise.accuracy.rmse(pred)

RMSE: 0.9282


np.float64(0.9282056050129103)

### Tuning for best k

In [17]:
from surprise.model_selection import GridSearchCV
from surprise.model_selection.split import KFold

In [66]:
param_grid = { 'k':[25,40,50,65,75,85],
             'user_based':[False,True]
             }
kfold = KFold(n_splits=5, random_state=26,shuffle=True)
# gs = GridSearchCV(surprise.KNNBasic, param_grid,measures=['rmse','mae'], cv=kfold)
gs = GridSearchCV(surprise.KNNWithZScore, param_grid,measures=['rmse','mae'], cv=kfold)
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [67]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.82858464566139
{'k': 85, 'user_based': False}
0.6262836041025638
{'k': 85, 'user_based': False}


### Building best model on whole data

In [45]:
algo = gs.best_estimator['rmse']
algo.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


In [50]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(a,b)  # -> elements of a which not present in b

array([2, 5, 8])

In [51]:
a = [2,4,5,7,8,9]
b = [4,7,9]
np.setdiff1d(b,a)  # -> elements of b which not present in a

array([], dtype=int64)

In [52]:
# Total Numbers of item
ItemID = df['ItemID'].unique()
len(ItemID)

2071

In [58]:
user = 50
u_iid = df[df['UserID'] == user ]['ItemID'].unique()
print('List of item rated by user :\n',u_iid)
print('No. of items rated by user {0} : {1}'.format(user,len(u_iid)))
iids_to_predict = np.setdiff1d(ItemID,u_iid)
print('Items not rated by the user or those items which the expected ratings are to be predicted :\n',iids_to_predict)

List of item rated by user :
 [  8 211   3   2 219 234  12 254 250 207  11 253 236  84  10   7 233  13
   1   5   6 252 241 216 257 206   4 217   9 215 213  17 255 220 121 245
 239 251 235]
No. of items rated by user 50 : 39
Items not rated by the user or those items which the expected ratings are to be predicted :
 [  14   15   16 ... 2069 2070 2071]


In [59]:
testset = [[user,iid,0.] for iid in iids_to_predict]

In [61]:
prediction = algo.test(testset)
prediction

[Prediction(uid=50, iid=np.int64(14), r_ui=0.0, est=np.float64(1.2275204359673026), details={'actual_k': 2, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(15), r_ui=0.0, est=np.float64(2.4154514276417527), details={'actual_k': 5, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(16), r_ui=0.0, est=np.float64(3.318980358997195), details={'actual_k': 4, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(18), r_ui=0.0, est=np.float64(3.2724795640326976), details={'actual_k': 2, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(19), r_ui=0.0, est=np.float64(2.544959128065395), details={'actual_k': 2, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(20), r_ui=0.0, est=np.float64(2.5586669192831555), details={'actual_k': 7, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(21), r_ui=0.0, est=np.float64(2.9487719250126854), details={'actual_k': 3, 'was_impossible': False}),
 Prediction(uid=50, iid=np.int64(22), r_ui=0.0, est=np.fl

In [65]:
exp_rating = [ (prediction[i].iid,prediction[i].est) for i in range(0,len(prediction)) ]
df_rating = pd.DataFrame(exp_rating , columns=['ItemID','Est_Rating'])
df_rating.sort_values('Est_Rating',ascending=False).head(30)

,ItemID,Est_Rating
2014,2054,4.0
814,854,4.0
2009,2049,4.0
827,867,4.0
2005,2045,4.0
786,826,4.0
787,827,4.0
2018,2058,4.0
632,672,4.0
630,670,4.0


# Exercise

In [70]:
df = pd.read_csv(r"C:\Users\PGCP-AI.STUDENTSDC\Desktop\Machine_Learning\Cases_Rec_Sys\ml-100k\u.data",
                 sep='\t',names=['UserId','ItemID','Rating','TS'])
df.sample(5)

,UserId,ItemID,Rating,TS
84764,381,118,1,892697051
65259,622,511,4,882592103
55366,457,1028,3,882393828
18188,62,462,2,879373737
88851,711,958,5,876278721


In [71]:
df.drop('TS',axis=1,inplace=True)

In [72]:
df.sample(2)

,UserId,ItemID,Rating
85753,937,326,1
46766,645,203,4


In [73]:
min_rate = df['Rating'].min()
max_rate = df['Rating'].max()

In [76]:
reader = surprise.Reader(rating_scale=(min_rate,max_rate))
data = surprise.Dataset.load_from_df(df,reader)

In [77]:
param_gird = {
    'k':[25,40,50,60,65,70,85],
    'user_based':[True,False]
}
kfold = KFold(n_splits=5,shuffle=True,random_state=26)
gs = GridSearchCV(surprise.KNNBasic,param_grid,cv=kfold,measures=['rmse','mae'])
gs.fit(data)

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

In [78]:
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])
print(gs.best_score['mae'])
print(gs.best_params['mae'])

0.9761915086082429
{'k': 25, 'user_based': False}
0.7698381577357556
{'k': 25, 'user_based': False}


In [82]:
model = gs.best_estimator['rmse']
model.fit(data.build_full_trainset())

Computing the msd similarity matrix...
Done computing similarity matrix.


In [83]:
ItemID = df['ItemID'].unique()
len(ItemID)

1682